# Morphology Analysis

**Author:** Kevin Druciak (kevintdruciak@gmail.com)

Quantitative analysis of segmented neurons: per-neuron volume, surface area, equivalent diameter, and sphericity. Includes size distribution plots, volume-vs-surface-area scatter, and 3D centroid visualization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, distance_transform_edt, binary_fill_holes
from skimage.exposure import equalize_adapthist
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from skimage.measure import regionprops_table, marching_cubes, mesh_surface_area
from skimage.color import label2rgb
import pandas as pd

## Generate Segmented Data

In [ ]:
np.random.seed(42)
shape = (32, 128, 128)
voxel_spacing = np.array([0.05, 0.004, 0.004])  # micrometers (typical EM)

volume = np.random.randint(120, 220, size=shape, dtype=np.uint8)
for offset in range(2):
    volume[:, offset::32, :] = np.random.randint(20, 70, size=volume[:, offset::32, :].shape).astype(np.uint8)
    volume[:, :, offset::32] = np.random.randint(20, 70, size=volume[:, :, offset::32].shape).astype(np.uint8)

smoothed = gaussian_filter(volume.astype(np.float64), sigma=1.0)
equalized = equalize_adapthist(smoothed, kernel_size=8, clip_limit=0.03)
rescaled = (equalized * 255).astype(np.uint8)
membrane_mask = rescaled < 128
interior = ~membrane_mask
distance = distance_transform_edt(interior)
coords = peak_local_max(distance, min_distance=5, labels=interior.astype(int))
markers = np.zeros_like(membrane_mask, dtype=np.int32)
for i, c in enumerate(coords, start=1):
    markers[tuple(c)] = i
labels = watershed(-distance, markers, mask=interior)

for lbl in np.unique(labels):
    if lbl == 0:
        continue
    if (labels == lbl).sum() < 200:
        labels[labels == lbl] = 0

print(f"Segments for analysis: {len(np.unique(labels)) - 1}")

## Per-Neuron Statistics

In [ ]:
props = regionprops_table(
    labels,
    properties=("label", "area", "centroid", "bbox", "equivalent_diameter_area"),
    spacing=voxel_spacing,
)
df = pd.DataFrame(props)
df = df.rename(columns={
    "area": "volume_um3",
    "centroid-0": "centroid_z_um",
    "centroid-1": "centroid_y_um",
    "centroid-2": "centroid_x_um",
    "equivalent_diameter_area": "equiv_diameter_um",
})

surface_areas = []
for lbl in df["label"]:
    binary = (labels == lbl).astype(np.uint8)
    try:
        verts, faces, _, _ = marching_cubes(binary, level=0.5, spacing=voxel_spacing)
        sa = mesh_surface_area(verts, faces)
    except Exception:
        sa = 0.0
    surface_areas.append(sa)

df["surface_area_um2"] = surface_areas
df["sphericity"] = (
    (np.pi ** (1/3) * (6 * df["volume_um3"]) ** (2/3)) / df["surface_area_um2"]
).clip(0, 1)

print(f"Computed stats for {len(df)} neurons")
df.head(10)

## Size Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(df["volume_um3"], bins=20, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Volume (\u03bcm\u00b3)")
axes[0].set_ylabel("Count")
axes[0].set_title("Neuron Volume Distribution")

axes[1].hist(df["surface_area_um2"], bins=20, color="coral", edgecolor="white")
axes[1].set_xlabel("Surface Area (\u03bcm\u00b2)")
axes[1].set_ylabel("Count")
axes[1].set_title("Surface Area Distribution")

axes[2].hist(df["equiv_diameter_um"], bins=20, color="mediumseagreen", edgecolor="white")
axes[2].set_xlabel("Equivalent Diameter (\u03bcm)")
axes[2].set_ylabel("Count")
axes[2].set_title("Equivalent Diameter Distribution")

plt.tight_layout()
plt.show()

## Volume vs. Surface Area

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    df["volume_um3"], df["surface_area_um2"],
    c=df["sphericity"], cmap="viridis", s=40, edgecolors="gray", linewidths=0.5,
)
ax.set_xlabel("Volume (\u03bcm\u00b3)")
ax.set_ylabel("Surface Area (\u03bcm\u00b2)")
ax.set_title("Volume vs. Surface Area (colored by sphericity)")
plt.colorbar(scatter, ax=ax, label="Sphericity")
plt.tight_layout()
plt.show()

## 3D Centroid Distribution

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(
    df["centroid_x_um"], df["centroid_y_um"], df["centroid_z_um"],
    s=df["volume_um3"] / df["volume_um3"].max() * 200,
    c=df["label"], cmap="tab20", alpha=0.7, edgecolors="gray",
)
ax.set_xlabel("X (\u03bcm)")
ax.set_ylabel("Y (\u03bcm)")
ax.set_zlabel("Z (\u03bcm)")
ax.set_title("Neuron Centroid Positions (size \u221d volume)")
plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
summary = df[["volume_um3", "surface_area_um2", "equiv_diameter_um", "sphericity"]].describe()
print(summary.to_string())

csv_path = "../output/neuron_stats.csv"
import os
os.makedirs(os.path.dirname(csv_path), exist_ok=True)
df.to_csv(csv_path, index=False)
print(f"\nExported to {csv_path}")